In [6]:
# ==============================================================================
# REPLICATION CODE: "Executives in Politics" 
# This script is written for easy reading. It loads Excel data from the "econ/" 
# folder, runs calculations, builds charts, and runs regression models.
# ==============================================================================

# 1. IMPORTING LIBRARIES
import pandas as pd                  # Used for data manipulation (like Excel in Python)
import matplotlib.pyplot as plt      # Used for drawing graphs and charts
from scipy import stats              # Used for statistical tests (like t-tests)
import statsmodels.formula.api as smf # Used for running regressions (OLS)

# Make our graphs look clean and academic
plt.style.use('seaborn-v0_8-whitegrid')

print("Starting Replication Process...\n")

# ==============================================================================
# PART 1: DESCRIPTIVE STATISTICS (Replicating Table 1)
# Goal: Show basic facts about how many politicians are executives and if they self-fund.
# ==============================================================================
print("--- PART 1: TABLE 1 SUMMARY STATISTICS ---")

# 1. Load BOTH parts of the Table 1 data
df_part1 = pd.read_excel('econ/Table1_Part1_BusinessPoliticians_PartyAffiliation.xlsx')
df_part2 = pd.read_excel('econ/Table1_Part2_SampleOfPoliticiansBasedOnOfficialBiographies.xlsx')

# 2. Merge them together based on the politician's unique ID (idbioguide)
# This gives us a single dataframe containing BOTH the 'BsnsPolitician' and 'REP_Flag' columns
df_pols = pd.merge(df_part1, df_part2, on='idbioguide', how='inner')

# Load the data for campaign self-funding
df_funding = pd.read_excel('econ/Table2_PanelA_Self_Funding.xlsx')

# Count the unique number of politicians in our merged dataset
total_pols = df_pols['idbioguide'].nunique()

# Count how many of them are flagged as "Business Politicians" (BsnsPolitician == 1)
bus_pols = df_pols[df_pols['BsnsPolitician'] == 1]['idbioguide'].nunique()

# Count how many business politicians are Republicans (REP_Flag == 1)
bus_reps = df_pols[(df_pols['BsnsPolitician'] == 1) & (df_pols['REP_Flag'] == 1)]['idbioguide'].nunique()

# Calculate the percentage
rep_percentage = (bus_reps / bus_pols) * 100

print(f"Total Unique Politicians: {total_pols}")
print(f"Total Business Politicians: {bus_pols}")
print(f"Percentage of Business Politicians who are Republican: {rep_percentage:.1f}%\n")

# Check self-funding: How many candidates gave more than $1 Million to their own campaign?
total_candidates = len(df_funding)
million_funders = len(df_funding[df_funding['above1Million_contri_loan'] == 1])
million_percentage = (million_funders / total_candidates) * 100

print(f"Percentage of candidates who self-funded over $1 Million: {million_percentage:.1f}%\n")

# ==============================================================================
# PART 2: DATA VISUALIZATION (Replicating Figures 1 to 5)
# ==============================================================================
print("-" * 50)
print("PART 2: GENERATING ALL FIGURES...")
print("-" * 50)

# --- FIGURE 1: Share of Business Politicians & Campaign Costs ---
df_costs = pd.read_excel('econ/Figure1_FECCampaignCosts.xlsx')
df_fed_pols = pd.read_excel('econ/Figure1_Figure2_PoliticiansInFederalOffice.xlsx')

avg_costs = df_costs.groupby('CAND_ELECTION_YR')['TotalSpending_AllCandFiles'].mean().reset_index()
pols_per_year = df_fed_pols.groupby('CAND_ELECTION_YR')['idbioguide'].count().reset_index()
pols_per_year.columns = ['Year', 'Total_Pols']
bus_per_year = df_fed_pols.groupby('CAND_ELECTION_YR')['SeniorExecutiveVerified'].sum().reset_index()
bus_per_year.columns = ['Year', 'Bus_Pols']

fig1_data = pd.merge(pols_per_year, bus_per_year, on='Year')
fig1_data['Share'] = fig1_data['Bus_Pols'] / fig1_data['Total_Pols']

fig, ax1 = plt.subplots(figsize=(10, 6))
ax1.plot(fig1_data['Year'], fig1_data['Share'], color='red', marker='o', label='Share of Business Pols')
ax1.set_xlabel('Election Year')
ax1.set_ylabel('Share of Business Politicians', color='red')

ax2 = ax1.twinx()
ax2.plot(avg_costs['CAND_ELECTION_YR'], avg_costs['TotalSpending_AllCandFiles'], color='blue', linestyle='--', marker='x')
ax2.set_ylabel('Average Campaign Cost ($)', color='blue')

plt.title('Figure 1: Rise of Business Politicians and Campaign Costs')
plt.savefig('Figure1.png')
plt.close()
print("Saved Figure 1 as 'Figure1.png'")


# --- FIGURE 2: Party Affiliation of Business Politicians ---
bus_pols_only = df_fed_pols[df_fed_pols['SeniorExecutiveVerified'] == 1]
party_counts = bus_pols_only.groupby(['CAND_ELECTION_YR', 'termsparty']).size().unstack(fill_value=0)
party_shares = party_counts.div(party_counts.sum(axis=1), axis=0)

party_shares.plot(kind='bar', stacked=True, figsize=(10, 6), color=['blue', 'red', 'green'])
plt.title('Figure 2: Party Affiliation of Business Politicians')
plt.xlabel('Election Year')
plt.ylabel('Share of Business Politicians')
plt.legend(title='Party', loc='upper left')
plt.savefig('Figure2.png')
plt.close()
print("Saved Figure 2 as 'Figure2.png'")


# --- FIGURE 3: Number of BoardEx Executives Running for Office ---
df_fig3 = pd.read_excel('econ/Figure3_BusinessPoliticiansFromBoardex.xlsx')
bus_runners = df_fig3[df_fig3['businessPoliticianFlag'] == 1]
execs_per_year = bus_runners.groupby('CAND_ELECTION_YR')['Director_ID'].nunique()

plt.figure(figsize=(10, 6))
plt.plot(execs_per_year.index, execs_per_year.values, color='red', marker='o', linewidth=2)
plt.title('Figure 3: Corporate Executives Running for Office')
plt.xlabel('Election Year')
plt.ylabel('Number of Executives Running')
plt.savefig('Figure3.png')
plt.close()
print("Saved Figure 3 as 'Figure3.png'")


# --- FIGURE 4: Early Fundraising Advantage ---
df_fig4 = pd.read_excel('econ/Figure4_CampaignFundraising.xlsx')
fig4_data = df_fig4.groupby(['tertile_ID', 'businessPoliticianFlag'])['tot_cand_amt'].mean().unstack()

fig4_data.plot(kind='bar', figsize=(8, 6), color=['lightblue', 'red'])
plt.title('Figure 4: Early Fundraising Advantage')
plt.xlabel('Campaign Stage (1 = Beginning, 2 = Middle, 3 = End)')
plt.ylabel('Average Contributions')
plt.legend(['Normal Politicians', 'Business Politicians'])
plt.savefig('Figure4.png')
plt.close()
print("Saved Figure 4 as 'Figure4.png'")


# --- FIGURE 5: Legislative Impact Over Time (Event Study) ---
df_fig5 = pd.read_excel('econ/Figure5_LegislativeImpact.xlsx')
scores =['COPE', 'CFA', 'CCUS', 'dwnom1']
titles =['Pro-Labor (COPE)', 'Pro-Consumer (CFA)', 'Pro-Business (CCUS)', 'Conservative (DW-NOMINATE)']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, score in enumerate(scores):
    score_data = df_fig5.groupby('YearsRelativeToTurnover')[score].mean().reset_index()
    ax = axes[i]
    ax.plot(score_data['YearsRelativeToTurnover'], score_data[score], marker='o', linestyle='--', color='tab:blue', label='Non-Business Pol')
    
    in_office = score_data[score_data['YearsRelativeToTurnover'].isin([0, 1, 2])]
    ax.plot(in_office['YearsRelativeToTurnover'], in_office[score], marker='o', linestyle='-', color='red', linewidth=2, label='Business Pol in Office')
    
    ax.set_title(f'Figure 5: {titles[i]}')
    ax.set_xlabel('Years Relative to Turnover (0 = Takes Office)')
    ax.set_ylabel('Average Voting Score')
    ax.set_xticks(range(-2, 5))

handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=2, fontsize=12)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig('Figure5.png')
plt.close()
print("Saved Figure 5 as 'Figure5.png'\n")


# ==============================================================================
# PART 3: REGRESSIONS & EVENT STUDIES (Replicating Tables 3, 6, 7, 8, 11)
# Goal: Use econometrics (Ordinary Least Squares) to prove causality.
# ==============================================================================
print("--- PART 3: REGRESSIONS AND STATISTICAL TESTS ---")

# --- TABLE 3: Do Business Politicians Raise More Early Money? ---
# Load first 90 days fundraising data
df_90days = pd.read_excel('econ/Table2_PanelB_PanelC_ContributionsReceivedInFirst90Days.xlsx')

# Drop missing values so the math doesn't break
df_90days = df_90days.dropna(subset=['tot_cand_amt', 'businessPoliticianFlag', 'REP_Flag', 'Incumbent_Flag'])

# We write our regression formula: "Predict Total Amount based on if they are a Business Pol, Republican, or Incumbent"
# C(CAND_ELECTION_YR) adds "Fixed Effects" for the year, meaning we control for yearly economic differences.
formula_t3 = 'tot_cand_amt ~ businessPoliticianFlag + REP_Flag + Incumbent_Flag + C(CAND_ELECTION_YR)'

# Run the OLS Regression
# We cluster standard errors by Candidate ID (CAND_ID) because the same politician might run multiple times.
model_t3 = smf.ols(formula_t3, data=df_90days).fit(cov_type='cluster', cov_kwds={'groups': df_90days['CAND_ID']})
print("\n--- Table 3: Early Fundraising Advantage ---")
# We print just the main variables to keep the output clean
print(model_t3.params[['businessPoliticianFlag', 'REP_Flag', 'Incumbent_Flag']])


# --- TABLE 7: Do companies' stocks go up when their executive wins? (Event Study) ---
# Load Cumulative Abnormal Returns (CAR) data
df_cars = pd.read_excel('econ/Table4_TablesB3_B4_B5_CARs.xlsx')

print("\n--- Table 7: Firm-Value Stock Returns (Event Study) ---")
print("Window (-1 to +x days) | Average Return | t-statistic | p-value")

# We loop through the 4 different time windows provided by the authors
windows =['CAR_Window_1', 'CAR_Window_2', 'CAR_Window_3', 'CAR_Window_4']
for window in windows:
    # Drop empty rows for this specific window
    clean_data = df_cars[window].dropna()
    
    # Calculate the average stock jump
    avg_return = clean_data.mean()
    
    # Run a 1-sample t-test to see if this jump is statistically different from zero (0)
    t_stat, p_val = stats.ttest_1samp(clean_data, 0)
    
    # Print the results nicely
    print(f"{window:<22} | {avg_return:.3%}       | {t_stat:.2f}        | {p_val:.3f}")


# --- TABLE 8: Do they get put on committees that regulate their old industries? ---
df_tab8 = pd.read_excel('econ/Table5_CommitteeAssignments.xlsx')
df_tab8 = df_tab8.dropna(subset=['assignment_cmts', 'same_industry_exp', 'seniority', 'powerful_cmt', 'Congress'])

# Formula: Predict getting a committee assignment based on past industry experience
formula_t8 = "assignment_cmts ~ same_industry_exp + seniority + powerful_cmt + C(Congress)"
model_t8 = smf.ols(formula_t8, data=df_tab8).fit(cov_type='cluster', cov_kwds={'groups': df_tab8['idbioguide']})

print("\n--- Table 8: Committee Assignments ---")
print(model_t8.params[['same_industry_exp', 'seniority', 'powerful_cmt']])


# --- TABLES 11 & 12: How do they vote? (Legislative Impact) ---
# CFA = Pro-consumer score, COPE = Pro-labor score, CCUS = Pro-business score
df_legis = pd.read_excel('econ/Table7_TableB7_LegislativeImpact_AllElections.xlsx')

# List of voting scores we want to test
voting_scores = ['CFA', 'COPE', 'CCUS', 'dwnom1']

df_legis = df_legis.dropna(subset=voting_scores +['SeniorExecutiveVerified', 'repFlag', 'RepublicanVoteShare', 'Cycle'])

print("\n--- Tables 11 & 12: Legislative Voting Impact ---")
# Loop through each voting score and run a regression
for score in voting_scores:
    formula_legis = f"{score} ~ SeniorExecutiveVerified + repFlag + RepublicanVoteShare + C(Cycle)"
    
    # Fit model, clustering by politician ID
    model_legis = smf.ols(formula_legis, data=df_legis).fit(cov_type='cluster', cov_kwds={'groups': df_legis['idbioguide']})
    
    print(f"\nTarget Variable: {score}")
    # Print the coefficient (the effect size) and the p-value (is it significant?)
    results = pd.DataFrame({
        'Effect Size (Coef)': model_legis.params,
        'P-Value': model_legis.pvalues
    }).loc[['SeniorExecutiveVerified', 'repFlag']]
    print(results)

print("\nReplication Complete! Check your folder for the saved .png charts.")
    

Starting Replication Process...

--- PART 1: TABLE 1 SUMMARY STATISTICS ---
Total Unique Politicians: 374
Total Business Politicians: 374
Percentage of Business Politicians who are Republican: 66.6%

Percentage of candidates who self-funded over $1 Million: 7.6%

--------------------------------------------------
PART 2: GENERATING ALL FIGURES...
--------------------------------------------------
Saved Figure 1 as 'Figure1.png'
Saved Figure 2 as 'Figure2.png'
Saved Figure 3 as 'Figure3.png'
Saved Figure 4 as 'Figure4.png'
Saved Figure 5 as 'Figure5.png'

--- PART 3: REGRESSIONS AND STATISTICAL TESTS ---

--- Table 3: Early Fundraising Advantage ---
businessPoliticianFlag    264.701168
REP_Flag                   25.524295
Incumbent_Flag            128.579284
dtype: float64

--- Table 7: Firm-Value Stock Returns (Event Study) ---
Window (-1 to +x days) | Average Return | t-statistic | p-value
CAR_Window_1           | 0.677%       | 5.30        | 0.000
CAR_Window_2           | 0.703%     